In [224]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import warnings

In [225]:
DATA_FILE = 'heart disease classification dataset.csv'
TARGET_COLUMN = 'target'
TEST_SIZE_RATIO = 0.30  
RANDOM_SEED = 7777
N_COMPONENTS_PCA = 8

In [226]:
try:
    # Load the dataset
    df = pd.read_csv(DATA_FILE)
    # The first column appears to be an index, drop it if it exists and is unnamed
    if df.columns[0] == df.columns[0]:
        df = df.iloc[:, 1:]
    
    print(f"Initial Shape: {df.shape}")
    print(f"Columns and Data Types:\n{df.info(verbose=False)}")
    
except FileNotFoundError:
    print(f"Error: The file '{DATA_FILE}' was not found.")
    exit()

Initial Shape: (303, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Columns: 14 entries, age to target
dtypes: float64(4), int64(8), object(2)
memory usage: 33.3+ KB
Columns and Data Types:
None


In [227]:
df = df.replace({'?': np.nan, '': np.nan, ' ': np.nan})
df = df.fillna(0)
print("Missing values filled with 0.")

Missing values filled with 0.


In [228]:
object_cols = df.select_dtypes(include='object').columns.tolist()

In [229]:
le = LabelEncoder()
df[TARGET_COLUMN] = le.fit_transform(df[TARGET_COLUMN])
print(f"Target variable '{TARGET_COLUMN}' encoded: {le.classes_} -> {le.transform(le.classes_)}")

Target variable 'target' encoded: ['no' 'yes'] -> [0 1]


In [230]:
if TARGET_COLUMN in object_cols:
    object_cols.remove(TARGET_COLUMN)

In [231]:
df = pd.get_dummies(df, columns=object_cols, drop_first=True)
print(f"Shape after One-Hot Encoding: {df.shape}")

Shape after One-Hot Encoding: (303, 14)


In [232]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

In [233]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE_RATIO, random_state=RANDOM_SEED, stratify=y
)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (212, 13)
X_test shape:  (91, 13)


In [234]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [235]:
N_COMPONENTS_PCA = min(N_COMPONENTS_PCA, X_train_scaled.shape[1])

In [236]:
pca = PCA(n_components=N_COMPONENTS_PCA)

# Fit PCA ONLY on the training data and transform both sets
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA-transformed Training shape: {X_train_pca.shape}")
print(f"PCA-transformed Testing shape:  {X_test_pca.shape}")

PCA-transformed Training shape: (212, 8)
PCA-transformed Testing shape:  (91, 8)


In [237]:
explained_variance = pca.explained_variance_ratio_
print(f"\nExplained Variance Ratio by each component:\n{explained_variance}")
print(f"Total Variance Explained: {explained_variance.sum():.2f}")


Explained Variance Ratio by each component:
[0.20373622 0.11427604 0.09744069 0.0935391  0.08052009 0.07791612
 0.06855669 0.06458251]
Total Variance Explained: 0.80


In [238]:
knn = KNeighborsClassifier(n_neighbors=5)

# Train the model using the PCA-transformed training data
knn.fit(X_train_pca, y_train)

# Make predictions on the PCA-transformed test data
y_pred = knn.predict(X_test_pca)
print("KNN model trained and predictions made on the test set.")

KNN model trained and predictions made on the test set.


In [239]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Accuracy:  0.7802
Precision: 0.7778
Recall:    0.8400
F1 Score:  0.8077


In [240]:
print("\nConfusion Matrix:")
# Rows are True (Actual) labels, Columns are Predicted labels
print(pd.DataFrame(conf_matrix, 
                   index=['Actual No Disease (0)', 'Actual Disease (1)'], 
                   columns=['Predicted No Disease (0)', 'Predicted Disease (1)']))


Confusion Matrix:
                       Predicted No Disease (0)  Predicted Disease (1)
Actual No Disease (0)                        29                     12
Actual Disease (1)                            8                     42
